# Avance 6 — Evaluación de Implementación y Propuesta de Producción

**Proyecto Integrador — Equipo 16**  
**Maestría en Inteligencia Artificial Aplicada (MNA)**  
**Fase:** CRISP-ML(Q) — Evaluación e Implementación

---

## Objetivos

4.1 Determinar si el modelo se puede implementar o si es necesario retroceder a fases anteriores.  
4.2 Exponer las medidas o decisiones específicas que deben tomarse en respuesta a los resultados.  
4.3 Proponer un entorno de producción confiable, escalable y eficiente.

In [1]:
# Cell 0 — Imports
import sys
import os
import json
import time
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import display, Markdown, HTML
import pandas as pd

# Ensure project root is in path
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

# AWS
import boto3

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version}")

Project root: /Users/davidlmz/Documents/MNA/PI/valkiria_auditor
Python: 3.14.5 (main, May 10 2026, 10:21:34) [Clang 21.0.0 (clang-2100.0.123.102)]


---
## 1. Resumen de Resultados Previos

### 1.1 Pipeline RAG (Avance 2–3)

| Métrica | Valor | Umbral Aceptable |
|---------|-------|------------------|
| Recall@5 | 0.85 | > 0.70 |
| Precision@5 | 0.89 | > 0.70 |
| MRR | 0.71 | > 0.50 |
| Latencia media | 68 ms | < 200 ms |
| Dimensiones embedding | 1024 (Titan V2) | — |
| Chunking | Semántico (95th percentile) | — |

### 1.2 Evaluación de Modelos (Avance 4)

| Modelo | Latencia (s) | Calidad (5-pt) | Costo/1K tok |
|--------|-------------|----------------|-------------|
| GPT OSS 120B | 4.2 | 4.6 | $0.0035 |
| Qwen3 32B | 2.1 | 4.1 | $0.0008 |
| Claude Sonnet 4.5 | 5.8 | 4.8 | $0.015 |
| Nova Pro | 3.5 | 4.3 | $0.0032 |
| DeepSeek R1 | 8.2 | 4.7 | $0.0055 |

### 1.3 Evaluación de Agentes (Avance 5)

- **11 modelos** evaluados con LLM-as-judge (Claude Sonnet 4.5)
- **3 agentes especializados** (Gobernanza, Organizacional, Técnico) sobre ISO 27001:2013
- **140 controles** evaluados con evidencia vía S3 Vectors + RAG
- Precisión promedio del juez: **4.1/5.0** para el mejor modelo (Claude Sonnet 4.5)
- Estructura JSON válida: **>95%** para modelos tier-1

---
## 2. Decisión de Implementación (Objetivo 4.1)

### ¿El rendimiento es suficiente para producción?

**Sí.** Los criterios de éxito definidos en la fase de planeación se cumplen:

| Criterio | Meta | Resultado | Veredicto |
|----------|------|-----------|----------|
| RAG Recall@5 | > 0.70 | 0.85 | ✅ Supera |
| RAG Precision@5 | > 0.70 | 0.89 | ✅ Supera |
| Estructura JSON válida | > 90% | > 95% | ✅ Supera |
| Cobertura de controles | 100% Annex A | 93/93 (2022), 140/140 (2013) | ✅ Completa |
| Latencia end-to-end | < 10 min | ~5–7 min (auditoría completa) | ✅ Dentro |
| Costo por auditoría | < $5 USD | ~$1.50–3.00 | ✅ Dentro |

### ¿Existe margen para mejorar?

Sí, en tres dimensiones:
1. **Fine-tuning (QLoRA):** Entrenar Ministral 3B o Qwen3-8B en datos de auditoría para reducir latencia 60% manteniendo calidad
2. **Embeddings 512d:** Reducir dimensionalidad de 1024 a 512 (99% precisión retenida, 50% ahorro en almacenamiento)
3. **Paralelización:** Los 4 agentes de dominio (A.5–A.8) pueden ejecutarse en paralelo (ya implementado en LangGraph)

### Conclusión

**El sistema es viable para implementación en producción.** No es necesario retroceder a fases anteriores. Los resultados superan los umbrales definidos y la arquitectura serverless permite escalar según demanda.

---
## 3. Medidas y Decisiones para Stakeholders (Objetivo 4.2)

### Recomendaciones Clave para Implementación

1. **Modelo primario:** GPT OSS 120B (open-weight, balance calidad/costo) para agentes de evaluación; Qwen3 32B para agentes de baja latencia (evidencia)
2. **Modelo juez:** Claude Sonnet 4.5 para validación de calidad en producción (muestreo 10% de auditorías)
3. **Vector store:** S3 Vectors en producción (cosine, 1024d float32) con filtrado por metadata
4. **Orquestación:** Strands Agents SDK como framework primario (menor overhead, integración nativa con Bedrock)

### Acciones para Stakeholders

| Stakeholder | Acción | Prioridad | Timeline |
|-------------|--------|-----------|----------|
| CISO | Validar controles evaluados contra SoA organizacional | Alta | Semana 1 |
| Equipo de Auditoría | Calibrar umbrales de conformidad con hallazgos reales | Alta | Semana 1–2 |
| IT Operations | Desplegar CDK stacks (Ingestion + Backend + Frontend) | Media | Semana 2–3 |
| Gestión de Riesgos | Mapear hallazgos a registro de riesgos existente | Media | Semana 3–4 |
| Dirección | Revisar reporte ejecutivo de primera auditoría piloto | Baja | Semana 4 |

---
## 4. Análisis Comparativo de Plataformas Cloud (Objetivo 4.3)

### 4.1 Evaluación de Proveedores

In [2]:
# Tabla comparativa de proveedores cloud
comparison_data = {
    "Criterio": [
        "Modelos Foundation disponibles",
        "Framework de agentes nativo",
        "Vector store serverless",
        "RAG gestionado (Knowledge Bases)",
        "Fine-tuning serverless",
        "Observabilidad nativa para LLMs",
        "Costo por 1M tokens (tier medio)",
        "Escalabilidad serverless",
        "Menor vendor lock-in (open-weight)",
        "Soporte multi-framework agéntico",
    ],
    "AWS (Bedrock)": [
        "20+ (Anthropic, Meta, Mistral, Cohere, Amazon, OpenAI, Qwen, DeepSeek)",
        "Bedrock AgentCore + Strands SDK",
        "S3 Vectors, OpenSearch Serverless",
        "Bedrock Knowledge Bases (chunking, parsing, embedding automático)",
        "SageMaker JumpStart + Bedrock Custom Model Import",
        "CloudWatch + AgentCore Observability",
        "$3–15 USD",
        "Lambda + Step Functions (auto-scale, pay-per-use)",
        "Alta: GPT OSS, Llama, Mistral, Qwen, DeepSeek (open-weight)",
        "Strands, LangGraph, CrewAI, AutoGen — todos desplegables",
    ],
    "Azure (OpenAI)": [
        "5–8 (GPT-4o, GPT-4, Phi, Mistral limitado)",
        "Azure AI Agent Service (preview)",
        "Azure AI Search (no serverless nativo)",
        "Azure AI Search + Prompt Flow",
        "Azure ML Fine-tuning (provisioned)",
        "Azure Monitor + Application Insights",
        "$5–30 USD",
        "Azure Functions + Durable Functions",
        "Baja: dependencia de modelos OpenAI propietarios",
        "Semantic Kernel, LangChain — ecosistema más cerrado",
    ],
    "GCP (Vertex AI)": [
        "8–12 (Gemini, PaLM, Claude, Llama)",
        "Vertex AI Agents (preview)",
        "Vertex AI Vector Search (provisioned)",
        "Vertex AI Search + RAG Engine",
        "Vertex AI Model Tuning",
        "Cloud Monitoring + Vertex AI Experiments",
        "$3.50–20 USD",
        "Cloud Functions + Workflows",
        "Media: Gemini propietario, pero Llama/Claude disponibles",
        "LangChain, Vertex AI SDK — menos flexible",
    ],
    "IBM Watson": [
        "3–5 (Granite, Llama limitado)",
        "Watson Orchestrate (enterprise)",
        "Watson Discovery (provisioned)",
        "Watson Discovery + Assistant",
        "Watson Studio (manual)",
        "IBM Instana",
        "$10–40 USD",
        "IBM Cloud Functions (limitado)",
        "Baja: ecosistema cerrado Granite",
        "Limitado a Watson SDK",
    ],
}

df_comparison = pd.DataFrame(comparison_data)
display(df_comparison.style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
))

,Criterio,AWS (Bedrock),Azure (OpenAI),GCP (Vertex AI),IBM Watson
0,Modelos Foundation disponibles,"20+ (Anthropic, Meta, Mistral, Cohere, Amazon, OpenAI, Qwen, DeepSeek)","5–8 (GPT-4o, GPT-4, Phi, Mistral limitado)","8–12 (Gemini, PaLM, Claude, Llama)","3–5 (Granite, Llama limitado)"
1,Framework de agentes nativo,Bedrock AgentCore + Strands SDK,Azure AI Agent Service (preview),Vertex AI Agents (preview),Watson Orchestrate (enterprise)
2,Vector store serverless,"S3 Vectors, OpenSearch Serverless",Azure AI Search (no serverless nativo),Vertex AI Vector Search (provisioned),Watson Discovery (provisioned)
3,RAG gestionado (Knowledge Bases),"Bedrock Knowledge Bases (chunking, parsing, embedding automático)",Azure AI Search + Prompt Flow,Vertex AI Search + RAG Engine,Watson Discovery + Assistant
4,Fine-tuning serverless,SageMaker JumpStart + Bedrock Custom Model Import,Azure ML Fine-tuning (provisioned),Vertex AI Model Tuning,Watson Studio (manual)
5,Observabilidad nativa para LLMs,CloudWatch + AgentCore Observability,Azure Monitor + Application Insights,Cloud Monitoring + Vertex AI Experiments,IBM Instana
6,Costo por 1M tokens (tier medio),$3–15 USD,$5–30 USD,$3.50–20 USD,$10–40 USD
7,Escalabilidad serverless,"Lambda + Step Functions (auto-scale, pay-per-use)",Azure Functions + Durable Functions,Cloud Functions + Workflows,IBM Cloud Functions (limitado)
8,Menor vendor lock-in (open-weight),"Alta: GPT OSS, Llama, Mistral, Qwen, DeepSeek (open-weight)",Baja: dependencia de modelos OpenAI propietarios,"Media: Gemini propietario, pero Llama/Claude disponibles",Baja: ecosistema cerrado Granite
9,Soporte multi-framework agéntico,"Strands, LangGraph, CrewAI, AutoGen — todos desplegables","Semantic Kernel, LangChain — ecosistema más cerrado","LangChain, Vertex AI SDK — menos flexible",Limitado a Watson SDK


In [7]:
# Scoring ponderado
scoring_data = {
    "Criterio": [
        "Disponibilidad de modelos",
        "Framework de agentes",
        "Vector store serverless",
        "RAG gestionado",
        "Costo operativo",
        "Escalabilidad",
        "Menor vendor lock-in",
        "Multi-framework",
    ],
    "Peso": [0.20, 0.15, 0.10, 0.15, 0.15, 0.10, 0.10, 0.05],
    "AWS": [5, 5, 5, 5, 4, 5, 5, 5],
    "Azure": [3, 3, 3, 4, 3, 4, 2, 3],
    "GCP": [4, 3, 3, 4, 4, 4, 3, 3],
    "IBM": [2, 2, 2, 3, 2, 2, 2, 1],
}

df_scoring = pd.DataFrame(scoring_data)

# Calcular scores ponderados
for provider in ["AWS", "Azure", "GCP", "IBM"]:
    df_scoring[f"{provider}_weighted"] = df_scoring[provider] * df_scoring["Peso"]

totals = {p: df_scoring[f"{p}_weighted"].sum() for p in ["AWS", "Azure", "GCP", "IBM"]}

print("\n" + "="*50)
print("SCORING FINAL (escala 1-5, ponderado)")
print("="*50)
for provider, score in sorted(totals.items(), key=lambda x: x[1], reverse=True):
    bar = "█" * int(score * 4)
    print(f"{provider:6s} │ {score:.2f}/5.00  {bar}")

print(f"\nProveedor seleccionado: AWS ({totals['AWS']:.2f}/5.00)")


SCORING FINAL (escala 1-5, ponderado)
AWS    │ 4.85/5.00  ███████████████████
GCP    │ 3.60/5.00  ██████████████
Azure  │ 3.15/5.00  ████████████
IBM    │ 2.10/5.00  ████████

Proveedor seleccionado: AWS (4.85/5.00)


### 4.2 Justificación de AWS

La elección de AWS se fundamenta en cinco factores:

1. **Flexibilidad de modelos:** Bedrock es el único servicio que ofrece acceso unificado a 20+ modelos de 8 proveedores (Anthropic, Meta, Mistral, Amazon, OpenAI, Qwen, DeepSeek, Cohere) bajo una misma API. Esto permite cambiar de modelo sin modificar código.

2. **Costo operativo:** La arquitectura 100% serverless (Lambda + Step Functions + S3 Vectors + API Gateway) elimina costos fijos. Se paga únicamente por invocación y token consumido. Costo estimado por auditoría completa: **$1.50–3.00 USD**.

3. **Escalabilidad:** Lambda escala de 0 a miles de ejecuciones concurrentes sin provisionar infraestructura. Step Functions coordina pipelines de larga duración. S3 Vectors y OpenSearch Serverless escalan almacenamiento automáticamente.

4. **Menor dependencia del proveedor:** AWS Bedrock prioriza modelos open-weight (GPT OSS 120B, Llama 4, Qwen3, Mistral, DeepSeek R1). Si se requiere migrar, estos modelos pueden desplegarse en cualquier infraestructura con GPU. Los embeddings (Titan V2, 1024d) tienen equivalentes open-source.

5. **AgentCore como framework de agentes:** Bedrock AgentCore proporciona primitivas de producción (Runtime, Memory STM/LTM, Gateway MCP) que permiten integrar múltiples frameworks multi-agente (Strands Agents, LangGraph, CrewAI) bajo una misma plataforma de despliegue, observabilidad y gestión de memoria.

---
## 5. Arquitectura de Producción Propuesta

```
┌────────────────────────────────────────────────────────────────────────────────┐
│                              USUARIO (Auditor Lead)                             │
└────────────────────────────────┬───────────────────────────────────────────────┘
                                 │ HTTPS
┌────────────────────────────────▼───────────────────────────────────────────────┐
│                         CloudFront CDN + S3 (Frontend)                          │
│                         Cognito (Autenticación)                                 │
└────────────────────────────────┬───────────────────────────────────────────────┘
                                 │ REST API
┌────────────────────────────────▼───────────────────────────────────────────────┐
│                         API Gateway (v1/audit/*)                                │
│                   POST /audit/{strands|langgraph|crewai|comp2013}               │
└───────┬───────────────┬───────────────┬───────────────┬───────────────────────┘
        │               │               │               │
        ▼               ▼               ▼               ▼
┌──────────────┐┌──────────────┐┌──────────────┐┌──────────────┐
│   Strands    ││  LangGraph   ││   CrewAI     ││  comp2013    │
│   Lambda     ││  Lambda      ││   Lambda     ││  Lambda      │
│  (10 min)    ││  (10 min)    ││  (10 min)    ││  (10 min)    │
└──────┬───────┘└──────┬───────┘└──────┬───────┘└──────┬───────┘
       │               │               │               │
       └───────────────┼───────────────┼───────────────┘
                       ▼               ▼
          ┌────────────────────────────────────────┐
          │         Amazon Bedrock                  │
          │  GPT OSS 120B │ Qwen3 32B │ Titan V2   │
          └────────────────────┬───────────────────┘
                               │
          ┌────────────────────▼───────────────────┐
          │    Bedrock Knowledge Bases (RAG)        │
          │  S3 Vectors │ OpenSearch Serverless     │
          └────────────────────────────────────────┘
                               │
          ┌────────────────────▼───────────────────┐
          │         Observabilidad                  │
          │  CloudWatch │ DynamoDB │ SNS Alerts     │
          └────────────────────────────────────────┘
```

### Stacks CDK Implementados

| Stack | Recursos | Estado |
|-------|----------|--------|
| `IngestionStack` | S3, OpenSearch Serverless, Step Functions, 4 Lambdas, API Gateway | Implementado |
| `BackendStack` | 4 Lambdas orquestadores, DynamoDB (2 tablas), API Gateway, CloudWatch Dashboard, SNS Alerts | Implementado |
| `FrontendStack` | S3, CloudFront, Cognito User Pool | Implementado |

---
## 6. Demostración del Sistema Multi-Agente

En esta sección se ejecuta el sistema multi-agente sobre una organización ficticia para demostrar la funcionalidad end-to-end.

In [4]:
# Configuración de la organización a auditar
ORGANIZATION_CONTEXT = (
    "TechSecure S.A. de C.V., empresa de tecnología financiera (fintech) con 150 empleados "
    "ubicada en Monterrey, México. Procesa transacciones de pago electrónico y almacena "
    "datos financieros sensibles de 50,000 clientes. Opera infraestructura en AWS con "
    "3 centros de datos virtuales (us-east-1, us-west-2, eu-west-1). "
    "Busca certificación ISO 27001:2022 por primera vez."
)

AUDIT_TYPE = "certification"
AUDIT_ID = f"avance6_demo_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"

print(f"Organización: TechSecure S.A. de C.V.")
print(f"Tipo de auditoría: {AUDIT_TYPE}")
print(f"Audit ID: {AUDIT_ID}")

Organización: TechSecure S.A. de C.V.
Tipo de auditoría: certification
Audit ID: avance6_demo_20260615_005548


### 6.1 Ejecución con Strands Agents (Framework Primario)

In [5]:
from backend.strandsagents.orchestrator import run_audit as strands_run_audit

print("Ejecutando auditoría con Strands Agents...")
print("Esto puede tomar 3–7 minutos.\n")

start_time = time.perf_counter()

strands_result = strands_run_audit(
    organization_context=ORGANIZATION_CONTEXT,
    audit_type=AUDIT_TYPE,
    audit_id=f"strands_{AUDIT_ID}",
    session_id=f"strands_session_{AUDIT_ID}",
)

strands_time = time.perf_counter() - start_time
print(f"\n✅ Completado en {strands_time:.1f}s")
print(f"Longitud del reporte: {len(strands_result.get('report', ''))} caracteres")

Ejecutando auditoría con Strands Agents...
Esto puede tomar 3–7 minutos.

{"timestamp": "2026-06-15T00:55:49.087175+00:00", "level": "INFO", "agent_name": "orchestration_flow", "message": "Orchestration flow started", "trace_id": "130452c894a04b1d88fc215581fb4309", "span_id": "02d07e5b1aa344cf", "parent_span_id": "", "audit_id": "strands_avance6_demo_20260615_005548", "session_id": "strands_session_avance6_demo_20260615_005548", "flow_id": "4e91d06d94ec"}
{"timestamp": "2026-06-15T00:55:49.344213+00:00", "level": "INFO", "agent_name": "consolidation_agent", "message": "Node orchestration started", "trace_id": "130452c894a04b1d88fc215581fb4309", "span_id": "02d07e5b1aa344cf", "parent_span_id": "", "audit_id": "strands_avance6_demo_20260615_005548", "session_id": "strands_session_avance6_demo_20260615_005548", "node_id": "orchestration", "model_id": "openai.gpt-oss-120b-1:0"}
We need to orchestrate steps: plan_audit, collect_evidence, evaluate each domain, then produce final report. We'l

ValueError: max() iterable argument is empty

In [ ]:
# Mostrar métricas de flujo
flow_metrics = strands_result.get("flow_metrics", {})

print("=" * 60)
print("MÉTRICAS DE ORQUESTACIÓN — STRANDS AGENTS")
print("=" * 60)
print(f"Tiempo total del flujo: {flow_metrics.get('total_flow_time_ms', 0)/1000:.1f}s")
print(f"Agentes invocados: {flow_metrics.get('total_agents_invoked', 0)}")
print(f"Total tokens: {flow_metrics.get('total_tokens', 0):,}")
print(f"Costo estimado: ${flow_metrics.get('total_cost_usd', 0):.4f} USD")
print(f"Nodo cuello de botella: {flow_metrics.get('bottleneck_node', 'N/A')}")
print(f"Razón: {flow_metrics.get('bottleneck_reason', 'N/A')}")

In [ ]:
# Mostrar reporte ejecutivo (primeros 3000 caracteres)
report = strands_result.get("report", "")
display(Markdown(f"### Reporte de Auditoría (extracto)\n\n{report[:3000]}"))

In [ ]:
# Diagrama Mermaid del flujo
mermaid = strands_result.get("mermaid_diagram", "")
if mermaid:
    display(Markdown(f"### Diagrama de Flujo de Orquestación\n\n{mermaid}"))

### 6.2 Ejecución con LangGraph (Comparativa)

In [ ]:
from backend.langgraph.orchestrator import run_audit as langgraph_run_audit

print("Ejecutando auditoría con LangGraph...")

start_time = time.perf_counter()

langgraph_result = langgraph_run_audit(
    organization_context=ORGANIZATION_CONTEXT,
    audit_type=AUDIT_TYPE,
    audit_id=f"langgraph_{AUDIT_ID}",
    session_id=f"langgraph_session_{AUDIT_ID}",
)

langgraph_time = time.perf_counter() - start_time
print(f"\n✅ Completado en {langgraph_time:.1f}s")
print(f"Longitud del reporte: {len(langgraph_result.get('report', ''))} caracteres")

### 6.3 Ejecución con CrewAI (Comparativa)

In [ ]:
from backend.crewai.orchestrator import run_audit as crewai_run_audit

print("Ejecutando auditoría con CrewAI...")

start_time = time.perf_counter()

crewai_result = crewai_run_audit(
    organization_context=ORGANIZATION_CONTEXT,
    audit_type=AUDIT_TYPE,
    audit_id=f"crewai_{AUDIT_ID}",
    session_id=f"crewai_session_{AUDIT_ID}",
)

crewai_time = time.perf_counter() - start_time
print(f"\n✅ Completado en {crewai_time:.1f}s")
print(f"Longitud del reporte: {len(crewai_result.get('report', ''))} caracteres")

### 6.4 Comparación de Frameworks

In [ ]:
# Tabla comparativa de ejecución
comparison_results = {
    "Framework": ["Strands Agents", "LangGraph", "CrewAI"],
    "Versión": ["1.20.0", "0.4.7", "1.2.0"],
    "Tiempo (s)": [round(strands_time, 1), round(langgraph_time, 1), round(crewai_time, 1)],
    "Reporte (chars)": [
        len(strands_result.get("report", "")),
        len(langgraph_result.get("report", "")),
        len(crewai_result.get("report", "")),
    ],
    "Éxito": [
        "✅" if strands_result.get("flow_metrics", {}).get("success") else "❌",
        "✅" if langgraph_result.get("success") else "❌",
        "✅" if not crewai_result.get("flow_metrics", {}).get("failed_nodes") else "❌",
    ],
}

df_results = pd.DataFrame(comparison_results)
display(df_results)

### 6.5 Ejecución del Sistema comp2013 (Evaluación con S3 Vectors + RAG)

In [ ]:
from backend.comp2013.orchestrator import run_audit as comp2013_run_audit

print("Ejecutando auditoría ISO 27001:2013 (agente Gobernanza, cláusula 9.3)...")
print("Este agente utiliza S3 Vectors para RAG + LLM evaluación.\n")

start_time = time.perf_counter()

comp2013_result = comp2013_run_audit(
    organization_context=ORGANIZATION_CONTEXT,
    audit_type=AUDIT_TYPE,
    audit_id=f"comp2013_{AUDIT_ID}",
    agents_in_scope=["gobernanza"],
    controls_in_scope=["9.3"],
    run_judge=True,
)

comp2013_time = time.perf_counter() - start_time

print(f"✅ Completado en {comp2013_time:.1f}s")
print(f"Controles evaluados: {comp2013_result.get('total_controls_evaluated', 0)}")
print(f"\nResultado del hallazgo:")
for finding in comp2013_result.get("findings", []):
    print(json.dumps(finding.get("finding", {}), indent=2, ensure_ascii=False)[:1500])

---
## 7. Infraestructura Desplegada (IaC)

Todo el sistema está definido como Infrastructure as Code con AWS CDK (TypeScript).

In [ ]:
# Resumen de recursos desplegados por stack
infra_data = {
    "Stack": [
        "IngestionStack", "IngestionStack", "IngestionStack", "IngestionStack",
        "IngestionStack", "IngestionStack",
        "BackendStack", "BackendStack", "BackendStack", "BackendStack",
        "BackendStack", "BackendStack",
        "FrontendStack", "FrontendStack", "FrontendStack",
    ],
    "Recurso": [
        "S3 Bucket (datos)", "OpenSearch Serverless", "Step Functions",
        "Lambda EDA", "Lambda KB Ingestion", "Lambda RAG Validation",
        "Lambda Strands", "Lambda LangGraph", "Lambda CrewAI", "Lambda comp2013",
        "DynamoDB (2 tablas)", "CloudWatch Dashboard",
        "S3 + CloudFront", "Cognito User Pool", "API Gateway",
    ],
    "Tipo": [
        "Storage", "Vector DB", "Orchestration",
        "Compute", "Compute", "Compute",
        "Compute", "Compute", "Compute", "Compute",
        "Database", "Monitoring",
        "CDN", "Auth", "API",
    ],
    "Costo Mensual Estimado": [
        "$0.02/GB", "$0.24/OCU-hr", "$0.025/1K transiciones",
        "Pay-per-use", "Pay-per-use", "Pay-per-use",
        "Pay-per-use", "Pay-per-use", "Pay-per-use", "Pay-per-use",
        "Pay-per-use", "$3/dashboard",
        "$0.085/GB", "$0.0055/MAU", "$3.50/M requests",
    ],
}

df_infra = pd.DataFrame(infra_data)
display(df_infra.style.hide(axis="index"))

### Estimación de Costo Mensual en Producción

Suponiendo **50 auditorías/mes** con un promedio de 100K tokens por auditoría:

| Componente | Costo Estimado |
|------------|---------------|
| Bedrock (modelos) | $75–150 USD |
| Lambda (compute) | $5–10 USD |
| S3 Vectors + OpenSearch | $15–30 USD |
| DynamoDB | $2–5 USD |
| CloudFront + S3 (frontend) | $3–5 USD |
| CloudWatch | $5–10 USD |
| **Total** | **$105–210 USD/mes** |

Comparado con un auditor humano ($150–300 USD/hora × 40+ horas = $6,000–12,000 por auditoría), el sistema reduce costos en **>95%** por auditoría mientras mantiene cobertura completa de 93 controles.

---
## 8. Observabilidad del Sistema

El sistema implementa observabilidad en tres niveles:

In [ ]:
# Demostrar el sistema de observabilidad
from backend.observability.flow_tracker import FlowTracker

# Simular un flujo con métricas
demo_tracker = FlowTracker(
    audit_id="demo_observability",
    session_id="demo_session",
    actor_id="auditor_demo",
    publish_metrics=False,  # No publicar a CloudWatch en demo
)

# Simular nodos del flujo
import time

with demo_tracker.node("planning", agent_name="planning_agent", model_id="openai.gpt-oss-120b-1:0") as node:
    time.sleep(0.1)  # Simulación
    node.set_result(input_tokens=500, output_tokens=1500, tool_calls=2, cost=0.003)

with demo_tracker.node("evidence", agent_name="evidence_agent", model_id="qwen.qwen3-32b-v1:0") as node:
    time.sleep(0.05)
    node.set_result(input_tokens=800, output_tokens=2500, tool_calls=5, cost=0.001)

with demo_tracker.node("org_controls", agent_name="org_controls_agent", model_id="openai.gpt-oss-120b-1:0") as node:
    time.sleep(0.08)
    node.set_result(input_tokens=2000, output_tokens=4000, tool_calls=37, cost=0.008)

with demo_tracker.node("people_controls", agent_name="people_controls_agent", model_id="qwen.qwen3-32b-v1:0") as node:
    time.sleep(0.03)
    node.set_result(input_tokens=600, output_tokens=1200, tool_calls=8, cost=0.001)

with demo_tracker.node("tech_controls", agent_name="tech_controls_agent", model_id="openai.gpt-oss-120b-1:0") as node:
    time.sleep(0.07)
    node.set_result(input_tokens=1800, output_tokens=3500, tool_calls=34, cost=0.007)

with demo_tracker.node("consolidation", agent_name="consolidation_agent", model_id="openai.gpt-oss-120b-1:0") as node:
    time.sleep(0.06)
    node.set_result(input_tokens=3000, output_tokens=5000, tool_calls=3, cost=0.012)

# Obtener resumen
flow_summary = demo_tracker.finalize()

print("MÉTRICAS DE OBSERVABILIDAD (NIVEL 3: Orquestación)")
print("=" * 55)
print(f"Tiempo total: {flow_summary['total_flow_time_ms']:.0f} ms")
print(f"Ruta crítica: {flow_summary['critical_path_time_ms']:.0f} ms")
print(f"Agentes completados: {flow_summary['total_agents_invoked']}")
print(f"Tokens totales: {flow_summary['total_tokens']:,}")
print(f"Costo total: ${flow_summary['total_cost_usd']:.4f}")
print(f"Cuello de botella: {flow_summary['bottleneck_node']}")
print(f"\nNodos:")
for nid, info in flow_summary["nodes"].items():
    print(f"  {nid:20s} │ {info['duration_ms']:7.1f}ms │ {info['tokens']:5d} tok │ ${info['cost_usd']:.4f}")

In [ ]:
# Mostrar diagrama Mermaid del flujo
mermaid_diagram = demo_tracker.mermaid_diagram()
display(Markdown(f"### Diagrama de Flujo con Métricas\n\n{mermaid_diagram}"))

### Niveles de Observabilidad Implementados

| Nivel | Alcance | Métricas | Destino |
|-------|---------|----------|----------|
| 1 — Modelo | Por invocación LLM | Latencia, TTFT, OTPS, tokens, costo, throttles | CloudWatch |
| 2 — Agente | Por agente especializado | Ejecución, LLM time, tool time, RAG queries, éxito | CloudWatch |
| 3 — Orquestación | Flujo completo | Tiempo total, ruta crítica, bottleneck, paralelismo | CloudWatch + S3 |

---
## 9. Conclusiones

### Viabilidad de Implementación

El sistema multi-agente de auditoría ISO 27001 **es viable para producción**:
- Cumple todos los criterios de éxito definidos (RAG, cobertura, latencia, costo)
- Arquitectura 100% serverless lista para escalar
- Observabilidad completa en tres niveles
- Soporte para múltiples frameworks de orquestación

### Ventaja Competitiva de AWS

AWS fue seleccionado por su combinación de:
1. **20+ modelos** de 8 proveedores bajo API unificada (Bedrock)
2. **AgentCore** como plataforma de agentes con Memory, Runtime y Gateway
3. **Modelo open-weight** (GPT OSS 120B, Llama, Qwen) que minimiza vendor lock-in
4. **Costo operativo** ~$1.50–3.00 por auditoría completa
5. **Escalabilidad** de 0 a miles de auditorías concurrentes sin provisioning

### Próximos Pasos

1. Ejecutar auditoría piloto con documentación real del sponsor
2. Fine-tuning de Qwen3-8B con datos de evaluaciones validadas
3. Activar memoria a largo plazo (LTM) para contexto organizacional persistente
4. Integrar con sistema de gestión de riesgos del sponsor